## Persiapan: Membuat Dataset

In [1]:
import numpy as np
import pandas as pd

np.random.seed(111)
n = 2500
menu_list = ["Kopi Susu", "Americano", "Cappuccino", "Roti Bakar", "Kentang Goreng", "Es Teh"]
cabang_list = ["Magelang", "Yogyakarta", "Semarang"]
waktu_list = ["Pagi", "Siang", "Sore", "Malam"]

data_tugas11 = pd.DataFrame({
    "nama_menu": np.random.choice(menu_list, size=n),
    "cabang": np.random.choice(cabang_list, size=n),
    "waktu_pesan": np.random.choice(waktu_list, size=n),
    "qty": np.random.randint(1, 6, size=n),
    "harga_satuan": np.random.choice([15000, 18000, 20000, 22000, 25000, 30000], size=n),
})
data_tugas11["total_penjualan"] = data_tugas11["qty"] * data_tugas11["harga_satuan"] + np.random.normal(0, 8000, size=n)
data_tugas11.to_csv("tugas11_penjualan.csv", index=False)

print(f"Dataset berhasil dibuat: {len(data_tugas11)} baris")
print(data_tugas11.head())

Dataset berhasil dibuat: 2500 baris
        nama_menu      cabang waktu_pesan  qty  harga_satuan  total_penjualan
0  Kentang Goreng    Semarang       Malam    1         30000     27750.163532
1  Kentang Goreng  Yogyakarta        Sore    3         20000     64216.160744
2  Kentang Goreng    Semarang       Siang    5         22000    108722.792592
3  Kentang Goreng  Yogyakarta        Pagi    2         20000     54483.205872
4      Roti Bakar    Magelang       Malam    2         30000     58157.944120


## SparkSession & Membaca Data

In [2]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col

spark = SparkSession.builder \
    .appName("Tugas11-MLlibRegresi") \
    .master("local[*]") \
    .getOrCreate()
spark.sparkContext.setLogLevel("ERROR")
spark.conf.set("spark.sql.shuffle.partitions", "4")

df = spark.read.csv("tugas11_penjualan.csv", header=True, inferSchema=True)
df.printSchema()
df.show(5)
print("Total baris:", df.count())

26/11/12 12:57:21 WARN Utils: Your hostname, xcel resolves to a loopback address: 127.0.1.1; using 10.0.2.15 instead (on interface enp0s3)
26/11/12 12:57:21 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/11/12 12:57:22 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


root
 |-- nama_menu: string (nullable = true)
 |-- cabang: string (nullable = true)
 |-- waktu_pesan: string (nullable = true)
 |-- qty: integer (nullable = true)
 |-- harga_satuan: integer (nullable = true)
 |-- total_penjualan: double (nullable = true)

+--------------+----------+-----------+---+------------+------------------+
|     nama_menu|    cabang|waktu_pesan|qty|harga_satuan|   total_penjualan|
+--------------+----------+-----------+---+------------+------------------+
|Kentang Goreng|  Semarang|      Malam|  1|       30000|27750.163531558544|
|Kentang Goreng|Yogyakarta|       Sore|  3|       20000| 64216.16074371864|
|Kentang Goreng|  Semarang|      Siang|  5|       22000|108722.79259154423|
|Kentang Goreng|Yogyakarta|       Pagi|  2|       20000|54483.205871792496|
|    Roti Bakar|  Magelang|      Malam|  2|       30000| 58157.94412015849|
+--------------+----------+-----------+---+------------+------------------+
only showing top 5 rows

Total baris: 2500


## A. Feature Engineering

*StringIndexer* dipakai untuk mengubah *nama_menu*, *cabang*, dan *waktu_pesan* jadi indeks numerik. Ketiga indexer di-*fit()* sekali dan disimpan sebagai objek **Model** supaya bisa dipakai ulang di bagian E. Setelah itu seluruh fitur digabung jadi satu kolom *features* dengan *VectorAssembler*.

In [3]:
from pyspark.ml.feature import StringIndexer, VectorAssembler

# fit() -> mempelajari daftar kategori; hasilnya disimpan sebagai model indexer
idx_menu_model = StringIndexer(inputCol="nama_menu", outputCol="nama_menu_idx").fit(df)
idx_cabang_model = StringIndexer(inputCol="cabang", outputCol="cabang_idx").fit(df)
idx_waktu_model = StringIndexer(inputCol="waktu_pesan", outputCol="waktu_pesan_idx").fit(df)

# transform() -> menerapkan hasil pembelajaran ke data
df_indexed = idx_menu_model.transform(df)
df_indexed = idx_cabang_model.transform(df_indexed)
df_indexed = idx_waktu_model.transform(df_indexed)

df_indexed.select("nama_menu", "nama_menu_idx", "cabang", "cabang_idx",
                  "waktu_pesan", "waktu_pesan_idx").show(8)

# Gabungkan semua fitur menjadi satu kolom "features"
fitur_lengkap = ["qty", "harga_satuan", "nama_menu_idx", "cabang_idx", "waktu_pesan_idx"]
assembler = VectorAssembler(inputCols=fitur_lengkap, outputCol="features")

df_final = assembler.transform(df_indexed)
df_final.select("features", "total_penjualan").show(5, truncate=False)

+--------------+-------------+----------+----------+-----------+---------------+
|     nama_menu|nama_menu_idx|    cabang|cabang_idx|waktu_pesan|waktu_pesan_idx|
+--------------+-------------+----------+----------+-----------+---------------+
|Kentang Goreng|          5.0|  Semarang|       2.0|      Malam|            1.0|
|Kentang Goreng|          5.0|Yogyakarta|       1.0|       Sore|            3.0|
|Kentang Goreng|          5.0|  Semarang|       2.0|      Siang|            2.0|
|Kentang Goreng|          5.0|Yogyakarta|       1.0|       Pagi|            0.0|
|    Roti Bakar|          4.0|  Magelang|       0.0|      Malam|            1.0|
|     Americano|          0.0|  Magelang|       0.0|       Pagi|            0.0|
|    Cappuccino|          2.0|  Magelang|       0.0|      Malam|            1.0|
|    Cappuccino|          2.0|  Semarang|       2.0|       Pagi|            0.0|
+--------------+-------------+----------+----------+-----------+---------------+
only showing top 8 rows

+--

## B. Train/Test Split & Pelatihan Model

Data dibagi 75% training dan 25% testing dengan *seed=101*, lalu model *LinearRegression* dilatih dengan *labelCol="total_penjualan"*.

In [4]:
from pyspark.ml.regression import LinearRegression

train, test = df_final.randomSplit([0.75, 0.25], seed=101)
print(f"Jumlah data training : {train.count()}")
print(f"Jumlah data testing  : {test.count()}")

lr = LinearRegression(featuresCol="features", labelCol="total_penjualan")
model = lr.fit(train)

print("\nModel berhasil dilatih!")
print("Urutan fitur :", fitur_lengkap)
print("Koefisien    :", model.coefficients)
print("Intercept    :", model.intercept)

Jumlah data training : 1839
Jumlah data testing  : 661

Model berhasil dilatih!
Urutan fitur : ['qty', 'harga_satuan', 'nama_menu_idx', 'cabang_idx', 'waktu_pesan_idx']
Koefisien    : [22046.663271025758,3.022920868160672,-123.21425598657399,279.28452461677676,339.6387953231708]
Intercept    : -66903.17766485889


## C. Evaluasi Model

RMSE dan R² dihitung pada **data testing**.

In [5]:
from pyspark.ml.evaluation import RegressionEvaluator

predictions = model.transform(test)

ev_rmse = RegressionEvaluator(labelCol="total_penjualan", predictionCol="prediction", metricName="rmse")
ev_r2 = RegressionEvaluator(labelCol="total_penjualan", predictionCol="prediction", metricName="r2")

rmse = ev_rmse.evaluate(predictions)
r2 = ev_r2.evaluate(predictions)

print(f"RMSE : Rp {rmse:,.0f}")
print(f"R²   : {r2:.4f}  ({r2*100:.1f}% variasi data berhasil dijelaskan model)")

# Gambaran skala target sebagai pembanding RMSE
rata2 = df.agg({"total_penjualan": "mean"}).collect()[0][0]
print(f"\nRata-rata total_penjualan : Rp {rata2:,.0f}")
print(f"RMSE terhadap rata-rata   : {rmse/rata2*100:.1f}%")

RMSE : Rp 10,901
R²   : 0.9107  (91.1% variasi data berhasil dijelaskan model)

Rata-rata total_penjualan : Rp 65,174
RMSE terhadap rata-rata   : 16.7%


### Interpretasi

Model ini cukup baik: R² sebesar 0,9107 berarti sekitar 91% variasi *total_penjualan* dapat dijelaskan oleh model, dan RMSE sekitar Rp10.901 atau sekitar 16,7% dari rata-rata penjualan per pesanan (Rp65.174). Untuk perencanaan stok yang bersifat estimasi, model ini layak dipakai, tetapi prediksi per pesanan masih bisa meleset cukup terasa karena hubungan aslinya adalah perkalian *qty × harga_satuan*, sedangkan regresi linear hanya menjumlahkan pengaruh tiap fitur.

## D. Eksperimen Perbandingan Fitur

Model kedua hanya memakai *qty* dan *harga_satuan*. Supaya perbandingannya adil, model ini dilatih dan diuji pada **train/test split yang sama** dengan model utama.

In [6]:
# Model kedua: hanya qty & harga_satuan
assembler_dasar = VectorAssembler(inputCols=["qty", "harga_satuan"], outputCol="features_dasar")
train_d = assembler_dasar.transform(train)
test_d = assembler_dasar.transform(test)

lr_dasar = LinearRegression(featuresCol="features_dasar", labelCol="total_penjualan")
model_dasar = lr_dasar.fit(train_d)

pred_dasar = model_dasar.transform(test_d)
rmse_dasar = ev_rmse.evaluate(pred_dasar)
r2_dasar = ev_r2.evaluate(pred_dasar)

print("Koefisien model dasar (qty, harga_satuan):", model_dasar.coefficients)
print("Intercept model dasar:", model_dasar.intercept)

# Tabel perbandingan
hasil = spark.createDataFrame([
    ("Model utama (5 fitur)", float(r2), float(rmse)),
    ("Model dasar (qty + harga_satuan)", float(r2_dasar), float(rmse_dasar)),
], ["model", "R2", "RMSE"])
hasil.show(truncate=False)

print(f"Selisih R² (utama - dasar) : {r2 - r2_dasar:+.6f}")
print(f"Selisih RMSE (utama - dasar): Rp {rmse - rmse_dasar:+,.0f}")

Koefisien model dasar (qty, harga_satuan): [22048.50478817013,3.021049809988266]
Intercept model dasar: -66397.03118601872
+--------------------------------+------------------+------------------+
|model                           |R2                |RMSE              |
+--------------------------------+------------------+------------------+
|Model utama (5 fitur)           |0.910679109181288 |10900.803581280325|
|Model dasar (qty + harga_satuan)|0.9107572632054388|10896.033543172933|
+--------------------------------+------------------+------------------+

Selisih R² (utama - dasar) : -0.000078
Selisih RMSE (utama - dasar): Rp +5


### Kesimpulan

Menambahkan informasi menu, cabang, dan waktu pesan **tidak meningkatkan akurasi secara berarti**. R² model utama (5 fitur) adalah 0,910679, sedangkan model kedua (hanya *qty* dan *harga_satuan*) justru sedikit lebih tinggi, yaitu 0,910757 (selisih -0,00008), dan RMSE keduanya nyaris sama (selisih sekitar Rp5). Koefisien ketiga fitur hasil indexing juga sangat kecil dibanding *qty* (sekitar -123, 279, dan 340 berbanding 22.046 untuk *qty*). Hal ini wajar karena pada dataset ini *total_penjualan* memang dibangkitkan dari *qty × harga_satuan* ditambah noise, sehingga menu/cabang/waktu tidak membawa informasi tambahan. Selain itu, indeks hasil *StringIndexer* hanyalah label urutan frekuensi, bukan ukuran yang bermakna. Jadi untuk kasus ini, model dua fitur sudah cukup dan lebih sederhana.

## E. Prediksi Skenario Baru

Tiga skenario pesanan dummy dibuat sebagai DataFrame baru, lalu diproses dengan **model *StringIndexer* yang sama** (*idx_menu_model*, *idx_cabang_model*, *idx_waktu_model* dari bagian A) dan *assembler* yang sama, kemudian diprediksi menggunakan model utama.

In [7]:
skenario = pd.DataFrame({
    "nama_menu": ["Kopi Susu", "Roti Bakar", "Es Teh"],
    "cabang": ["Magelang", "Yogyakarta", "Semarang"],
    "waktu_pesan": ["Pagi", "Sore", "Malam"],
    "qty": [3, 5, 2],
    "harga_satuan": [22000, 30000, 15000],
})
df_baru = spark.createDataFrame(skenario)

# Pakai MODEL indexer yang sama (bukan StringIndexer baru) + assembler yang sama
df_baru = idx_menu_model.transform(df_baru)
df_baru = idx_cabang_model.transform(df_baru)
df_baru = idx_waktu_model.transform(df_baru)
df_baru = assembler.transform(df_baru)

pred_baru = model.transform(df_baru)
pred_baru.select("nama_menu", "cabang", "waktu_pesan", "qty", "harga_satuan",
                 col("prediction").alias("prediksi_total_penjualan")).show(truncate=False)

# Pembanding kasar: qty x harga_satuan
for _, r in skenario.iterrows():
    print(f"{r['nama_menu']:<11} | {r['cabang']:<10} | {r['waktu_pesan']:<5} -> qty x harga = Rp {r['qty']*r['harga_satuan']:,.0f}")

+----------+----------+-----------+---+------------+------------------------+
|nama_menu |cabang    |waktu_pesan|qty|harga_satuan|prediksi_total_penjualan|
+----------+----------+-----------+---+------------+------------------------+
|Kopi Susu |Magelang  |Pagi       |3  |22000       |65617.85699176662       |
|Roti Bakar|Yogyakarta|Sore       |5  |30000       |134823.10862173006      |
|Es Teh    |Semarang  |Malam      |2  |15000       |23062.52697619972       |
+----------+----------+-----------+---+------------+------------------------+

Kopi Susu   | Magelang   | Pagi  -> qty x harga = Rp 66,000
Roti Bakar  | Yogyakarta | Sore  -> qty x harga = Rp 150,000
Es Teh      | Semarang   | Malam -> qty x harga = Rp 30,000


### Catatan

Hasil prediksi untuk skenario 1 (Kopi Susu, Magelang, Pagi, qty 3, harga Rp22.000) sekitar Rp65.618, sangat dekat dengan hitungan *qty × harga* sebesar Rp66.000. Skenario 2 (Rp134.823 vs Rp150.000) dan skenario 3 (Rp23.063 vs Rp30.000) cenderung lebih rendah dari hitungan kasar tersebut. Ini konsisten dengan temuan di bagian C: model linear hanya mendekati hubungan perkalian *qty × harga*, sehingga prediksi untuk kombinasi ekstrem kurang presisi.

In [8]:
spark.stop()
print("SparkSession ditutup.")

SparkSession ditutup.
